In [0]:
WITH pricing_audit AS (
  SELECT 
    issue_bucket,
    sub_bucket,
    severity_score,
    supplier_id,
    segment,
    tour_id,
    tour_option_id,
    pricing_id,
    tour_title,
    tour_option_title,
    location_name,
    activity_type,
    tour_category,
    has_individual_pricing,
    has_group_pricing,
    pricing_type_used,
    problem_description,
    currency_iso_code,
    current_from_price,
    from_price_eur,
    exchange_rate_to_eur,
    benchmark_median_eur,
    benchmark_p95_eur,
    tour_min_from_price_eur,
    tour_max_from_price_eur,
    tour_price_spread_pct,
    base_retail_price,
    base_retail_price_eur,
    price_gap_pct
  FROM production.supply_analytics.option_price_discrepancy
),

tour_performance AS (
  SELECT 
    supplier_id,
    tour_id,
    tour_option_id,
    tour_title,
    tour_category,
    activity_type,
    location_name,
    bookings_l3mo,
    gmv_l3mo,
    nr_l3mo,
    tickets_l3mo,
    customers_l3mo,
    bookings_l6mo,
    gmv_l6mo,
    nr_l6mo,
    tickets_l6mo,
    customers_l6mo,
    bookings_l12mo,
    gmv_l12mo,
    nr_l12mo,
    tickets_l12mo,
    customers_l12mo,
    bookings_lifetime,
    gmv_lifetime,
    nr_lifetime,
    avg_booking_value_l12mo,
    avg_tickets_per_booking_l12mo,
    last_booking_date,
    first_booking_date,
    gmv_percentile,
    bookings_percentile,
    nr_percentile,
    basket_size_percentile,
    gmv_decile,
    gmv_quartile,
    gmv_tier,
    tour_value_tier,
    days_since_last_booking,
    booking_growth_trend_pct
  FROM production.supply_analytics.option_price_discrepancy_performance 
)

-- SIMPLE JOIN WITH IMPACT-BASED ORDERING
SELECT 
  pa.issue_bucket,
  pa.sub_bucket,
  pa.severity_score,
  pa.supplier_id,
  pa.segment,
  pa.tour_id,
  pa.tour_option_id,
  pa.pricing_id,
  pa.tour_title,
  pa.tour_option_title,
  pa.location_name,
  pa.activity_type,
  pa.tour_category,
  pa.has_individual_pricing,
  pa.has_group_pricing,
  pa.pricing_type_used,
  pa.problem_description,
  pa.currency_iso_code,
  pa.current_from_price,
  pa.from_price_eur,
  pa.exchange_rate_to_eur,
  pa.benchmark_median_eur,
  pa.benchmark_p95_eur,
  pa.tour_min_from_price_eur,
  pa.tour_max_from_price_eur,
  pa.tour_price_spread_pct,
  pa.base_retail_price,
  pa.base_retail_price_eur,
  pa.price_gap_pct,
  
  tp.bookings_l3mo,
  tp.gmv_l3mo,
  tp.nr_l3mo,
  tp.tickets_l3mo,
  tp.customers_l3mo,
  tp.bookings_l6mo,
  tp.gmv_l6mo,
  tp.nr_l6mo,
  tp.tickets_l6mo,
  tp.customers_l6mo,
  tp.bookings_l12mo,
  tp.gmv_l12mo,
  tp.nr_l12mo,
  tp.tickets_l12mo,
  tp.customers_l12mo,
  tp.bookings_lifetime,
  tp.gmv_lifetime,
  tp.nr_lifetime,
  tp.avg_booking_value_l12mo,
  tp.avg_tickets_per_booking_l12mo,
  tp.last_booking_date,
  tp.first_booking_date,
  tp.gmv_percentile,
  tp.bookings_percentile,
  tp.nr_percentile,
  tp.basket_size_percentile,
  tp.gmv_decile,
  tp.gmv_quartile,
  tp.gmv_tier,
  tp.tour_value_tier,
  tp.days_since_last_booking,
  tp.booking_growth_trend_pct
  
FROM pricing_audit pa
LEFT JOIN tour_performance tp
  ON pa.tour_option_id = tp.tour_option_id
  AND pa.tour_id = tp.tour_id
  AND pa.supplier_id = tp.supplier_id
WHERE pa.issue_bucket != 'BUCKET_1_HIGH_FROM_PRICE'
ORDER BY 
  pa.issue_bucket,
  pa.sub_bucket,
  COALESCE(tp.gmv_l12mo, 0) DESC, 
  COALESCE(tp.bookings_l12mo, 0) DESC,
  pa.severity_score DESC;
